In [2]:
from pathlib import Path

project_root = Path(r"C:\Projects\NeuroTrainer")
data_root = project_root / "data" / "external" / "muse_sleep_onset"

print("Project root:", project_root)
print("Dataset exists:", data_root.exists())

Project root: C:\Projects\NeuroTrainer
Dataset exists: True


In [3]:
subject = "sub-001"
session = "ses-001"

eeg_dir = (
    data_root
    / subject
    / session
    / "eeg"
)

print("EEG folder:", eeg_dir)
print("Exists:", eeg_dir.exists())

EEG folder: C:\Projects\NeuroTrainer\data\external\muse_sleep_onset\sub-001\ses-001\eeg
Exists: True


In [4]:
files = sorted(eeg_dir.iterdir())

for file in files:
    print(file.name)

sub-001_ses-001_space-CapTrak_coordsystem.json
sub-001_ses-001_space-CapTrak_electrodes.tsv
sub-001_ses-001_task-sleeponset_channels.tsv
sub-001_ses-001_task-sleeponset_eeg.eeg
sub-001_ses-001_task-sleeponset_eeg.json
sub-001_ses-001_task-sleeponset_eeg.vhdr
sub-001_ses-001_task-sleeponset_eeg.vmrk
sub-001_ses-001_task-sleeponset_events.json
sub-001_ses-001_task-sleeponset_events.tsv


In [5]:
import mne

vhdr_file = (
    eeg_dir
    / f"{subject}_{session}_task-sleeponset_eeg.vhdr"
)

raw = mne.io.read_raw_brainvision(
    vhdr_file,
    preload=False,
    verbose=False,
)

print("Channels:", raw.ch_names)
print("Sampling rate:", raw.info["sfreq"], "Hz")
print("Number of channels:", len(raw.ch_names))
print("Samples:", raw.n_times)
print(
    "Duration:",
    round(raw.n_times / raw.info["sfreq"] / 60, 2),
    "minutes"
)

Channels: ['TP9', 'AF7', 'AF8', 'TP10']
Sampling rate: 128.0 Hz
Number of channels: 4
Samples: 122880
Duration: 16.0 minutes


In [6]:
import csv

events_file = (
    eeg_dir
    / f"{subject}_{session}_task-sleeponset_events.tsv"
)

with events_file.open(
    encoding="utf-8-sig"
) as file:
    rows = list(
        csv.DictReader(
            file,
            delimiter="\t"
        )
    )

for row in rows:
    print(row)

{'onset': '660.0', 'duration': '0.0', 'trial_type': 'n2_onset', 'value': '1', 'sample': '84480'}


In [ ]:
n2_events = [row for row in rows if row["trial_type"] == "n2_onset"]
if len(n2_events) != 1:
    raise ValueError("Expected exactly one Muse n2_onset annotation")
n2_onset = float(n2_events[0]["onset"])

print("N2 onset:", n2_onset, "seconds")
print("N2 onset:", n2_onset / 60, "minutes")

In [8]:
window_seconds = 30
sfreq = raw.info["sfreq"]

start_sample = 0
stop_sample = int(window_seconds * sfreq)

X = raw.get_data(
    start=start_sample,
    stop=stop_sample
)

print("X shape:", X.shape)

X shape: (4, 3840)


In [9]:
current_time = window_seconds

y = n2_onset - current_time

print("Current time:", current_time, "seconds")
print("Seconds left to N2:", y)

Current time: 30 seconds
Seconds left to N2: 630.0


In [ ]:
window_seconds = 30
step_seconds = 30

sfreq = raw.info["sfreq"]

import sys
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.signal.sleep_onset import make_onset_row, MUSE_ONSET_RULE

recording = f"{subject}_{session}_task-sleeponset"
window_metadata = []
X_windows = []
y_targets = []

for current_time in range(
    window_seconds,
    int(n2_onset) + 1,
    step_seconds
):
    start_time = current_time - window_seconds

    start_sample = int(start_time * sfreq)
    stop_sample = int(current_time * sfreq)

    X = raw.get_data(
        start=start_sample,
        stop=stop_sample
    )

    row = make_onset_row(subject, recording, start_time, current_time,
                         n2_onset, MUSE_ONSET_RULE)
    if X.shape[1] != stop_sample - start_sample:
        raise ValueError("Incomplete EEG window")
    row["eeg_epoch_index"] = len(X_windows)
    window_metadata.append(row)
    y = row["time_to_n2_seconds"]

    X_windows.append(X)
    y_targets.append(y)

print("Кількість вікон:", len(X_windows))
print("Перше X shape:", X_windows[0].shape)
print("Перший y:", y_targets[0])
print("Останній y:", y_targets[-1])
# Explicit dataset definition and window timestamps for each EEG window.
import pandas as pd
window_metadata = pd.DataFrame(window_metadata)
print(window_metadata.head())


In [ ]:
# Save explicit window identities and timing without overwriting an earlier export.
metadata_dir = project_root / "data" / "processed" / "muse_sleep_onset"
metadata_dir.mkdir(parents=True, exist_ok=True)
metadata_path = metadata_dir / f"{recording}_windows.csv"
window_metadata.to_csv(metadata_path, index=False, mode="x")
print("Window metadata saved:", metadata_path)
